# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasperOwen/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

The rule in plain words:

Pages whose rankings have decreased should be reviewed for refresh first. Among these pages, those with high visibility should be prioritised as they will affect more people

Reason code: Ranking decline

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

import duckdb
from huggingface_hub import login

login(token=hf_token)

con = duckdb.connect()
con.sql("SET enable_http_metadata_cache=true;")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
from huggingface_hub import hf_hub_download

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

file_path

'/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet'

In [3]:
step_a_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- volume signal: clicks, both windows (SUM is safe here, no zero-quirk on clicks)
    SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS early_clicks,
    SUM(CASE WHEN report_date >  '2026-03-15' THEN gsc_clicks ELSE 0 END) AS late_clicks,

    -- visibility floor + CTR ingredient: impressions, both windows
    SUM(CASE WHEN report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS early_impressions,
    SUM(CASE WHEN report_date >  '2026-03-15' THEN gsc_impressions ELSE 0 END) AS late_impressions,

    -- position signal: AVG, both windows, excluding the position=0 data quirk
    AVG(CASE WHEN report_date <= '2026-03-15' AND gsc_avg_position > 0
             THEN gsc_avg_position ELSE NULL END) AS early_avg_position,
    AVG(CASE WHEN report_date >  '2026-03-15' AND gsc_avg_position > 0
             THEN gsc_avg_position ELSE NULL END) AS late_avg_position

FROM read_parquet('{file_path}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
"""

step_a = con.sql(step_a_query).df()
step_a.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,early_clicks,late_clicks,early_impressions,late_impressions,early_avg_position,late_avg_position
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2.0,0.0,379.0,79.0,4.095154,5.548106
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,10.0,13.0,2336.0,1607.0,4.319753,4.460415
2,client_65de48885f4ef01b,content_3c286ded8bd68120,7.0,8.0,904.0,1276.0,8.777106,8.135446
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,1.0,7.0,330.0,173.0,5.475501,5.632185
4,client_65de48885f4ef01b,content_ff867882e604fa96,0.0,0.0,24.0,0.0,2.850000,NaN


In [4]:
import numpy as np
import pandas as pd

def position_tier(pos):
    if pd.isna(pos) or pos == 0:
      return np.nan

    if pos <= 3:
        return 1
    elif pos <= 10:
        return 2
    elif pos <= 20:
        return 3
    elif pos <= 50:
        return 4
    else:
        return 5

step_a["early_position_tier"] = step_a["early_avg_position"].apply(position_tier)
step_a["late_position_tier"] = step_a["late_avg_position"].apply(position_tier)
step_a["tier_change"] = step_a["late_position_tier"] - step_a["early_position_tier"]

step_a[["client_hash_id", "content_hash_id", "early_avg_position", "late_avg_position",
        "early_position_tier", "late_position_tier", "tier_change"]].head(10)

,client_hash_id,content_hash_id,early_avg_position,late_avg_position,early_position_tier,late_position_tier,tier_change
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,4.095154,5.548106,2.0,2.0,0.0
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,4.319753,4.460415,2.0,2.0,0.0
2,client_65de48885f4ef01b,content_3c286ded8bd68120,8.777106,8.135446,2.0,2.0,0.0
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,5.475501,5.632185,2.0,2.0,0.0
4,client_65de48885f4ef01b,content_ff867882e604fa96,2.850000,NaN,1.0,NaN,NaN
5,client_65de48885f4ef01b,content_d6c71358297cfd6a,6.438580,2.673970,2.0,1.0,-1.0
6,client_65de48885f4ef01b,content_6cdc3980c61ba7e9,6.538462,7.226852,2.0,2.0,0.0
7,client_c182d11e4862a37d,content_a0e19c582cf792a7,6.211128,5.564499,2.0,2.0,0.0
8,client_c182d11e4862a37d,content_d926564dfe83536b,25.401721,26.916987,4.0,4.0,0.0
9,client_c182d11e4862a37d,content_f4a0e5c90b283626,15.097646,NaN,3.0,NaN,NaN


In [5]:
print(step_a[["tier_change"]].isna().sum())
print(len(step_a))
print(f"tier_change missing: {34852/len(step_a):.1%}")

tier_change    34852
dtype: int64
63856
tier_change missing: 54.6%


In [6]:
def position_change_bucket(change):
    if pd.isna(change):
        return np.nan
    elif change < 0:
        return "improved"
    elif change == 0:
        return "stable"
    else:
        return "declined"

step_a["position_change"] = step_a["tier_change"].apply(position_change_bucket)

In [7]:
position_signal_table = (
    step_a[step_a["position_change"].notna()]
    ["position_change"]
    .value_counts()
    .reset_index()
)

position_signal_table.columns = ["position_change", "n"]

position_signal_table

,position_change,n
0,stable,18507
1,declined,5683
2,improved,4814


Position change verdict: CONFIRMED

A meaningful subset of pages in the dataset are declining in ranking (19%), indicating that ranking decline is a reasonable signal for prioritising pages for refresh review.

In [8]:
def visibility_bucket(change):
    if pd.isna(change):
        return np.nan
    elif change <= 8:
        return "low"
    elif change > 8 and change <= 459:
        return "medium"
    else:
        return "high"

step_a["visibility_bucket"] = step_a["late_impressions"].apply(visibility_bucket)

In [9]:
visibility_table = (
    step_a[step_a["visibility_bucket"].notna()]
    ["visibility_bucket"]
    .value_counts()
    .reset_index()
)

visibility_table.columns = ["Visibility bucket", "n"]

visibility_table

,Visibility bucket,n
0,medium,31673
1,low,16231
2,high,15952


Visibility verdict: MIXED

By itself, the visibility of each page cannot tell us if a page is declining or not. However, when used alongside position change it can be used to help us decide which pages to prioritise for review, as pages with high visibility have greater search exposure, meaning they will benefit more from being reviewed for refresh.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
step_a["visibility_score"] = step_a["visibility_bucket"].map({
    "low": 1,
    "medium": 2,
    "high": 4
})

step_a

,client_hash_id,content_hash_id,early_clicks,late_clicks,early_impressions,late_impressions,early_avg_position,late_avg_position,early_position_tier,late_position_tier,tier_change,position_change,visibility_bucket,visibility_score
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2.0,0.0,379.0,79.0,4.095154,5.548106,2.0,2.0,0.0,stable,medium,2
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,10.0,13.0,2336.0,1607.0,4.319753,4.460415,2.0,2.0,0.0,stable,high,4
2,client_65de48885f4ef01b,content_3c286ded8bd68120,7.0,8.0,904.0,1276.0,8.777106,8.135446,2.0,2.0,0.0,stable,high,4
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,1.0,7.0,330.0,173.0,5.475501,5.632185,2.0,2.0,0.0,stable,medium,2
4,client_65de48885f4ef01b,content_ff867882e604fa96,0.0,0.0,24.0,0.0,2.850000,NaN,1.0,NaN,NaN,NaN,low,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63851,client_20259bd6705d81d4,content_32058aa8a2e4f4fb,0.0,0.0,0.0,9.0,NaN,2.555556,NaN,1.0,NaN,NaN,medium,2
63852,client_20259bd6705d81d4,content_526be944d717ab76,0.0,0.0,0.0,8.0,NaN,26.500000,NaN,4.0,NaN,NaN,low,1
63853,client_20259bd6705d81d4,content_20c613ae83bad2ca,0.0,1.0,0.0,64.0,NaN,1.890625,NaN,1.0,NaN,NaN,medium,2
63854,client_20259bd6705d81d4,content_d4f53cf222510f09,0.0,0.0,0.0,3.0,NaN,0.666667,NaN,1.0,NaN,NaN,low,1


In [11]:
def calculate_score(row):
    if row["tier_change"] > 0:
        return row["tier_change"] * row["visibility_score"]
    else:
        return 0

step_a["page_score"] = step_a.apply(calculate_score, axis=1)

step_a

,client_hash_id,content_hash_id,early_clicks,late_clicks,early_impressions,late_impressions,early_avg_position,late_avg_position,early_position_tier,late_position_tier,tier_change,position_change,visibility_bucket,visibility_score,page_score
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2.0,0.0,379.0,79.0,4.095154,5.548106,2.0,2.0,0.0,stable,medium,2,0.0
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,10.0,13.0,2336.0,1607.0,4.319753,4.460415,2.0,2.0,0.0,stable,high,4,0.0
2,client_65de48885f4ef01b,content_3c286ded8bd68120,7.0,8.0,904.0,1276.0,8.777106,8.135446,2.0,2.0,0.0,stable,high,4,0.0
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,1.0,7.0,330.0,173.0,5.475501,5.632185,2.0,2.0,0.0,stable,medium,2,0.0
4,client_65de48885f4ef01b,content_ff867882e604fa96,0.0,0.0,24.0,0.0,2.850000,NaN,1.0,NaN,NaN,NaN,low,1,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63851,client_20259bd6705d81d4,content_32058aa8a2e4f4fb,0.0,0.0,0.0,9.0,NaN,2.555556,NaN,1.0,NaN,NaN,medium,2,0.0
63852,client_20259bd6705d81d4,content_526be944d717ab76,0.0,0.0,0.0,8.0,NaN,26.500000,NaN,4.0,NaN,NaN,low,1,0.0
63853,client_20259bd6705d81d4,content_20c613ae83bad2ca,0.0,1.0,0.0,64.0,NaN,1.890625,NaN,1.0,NaN,NaN,medium,2,0.0
63854,client_20259bd6705d81d4,content_d4f53cf222510f09,0.0,0.0,0.0,3.0,NaN,0.666667,NaN,1.0,NaN,NaN,low,1,0.0


In [12]:
step_a[step_a["page_score"] > 0][
    ["tier_change", "visibility_bucket", "visibility_score", "page_score"]
].head(10)

,tier_change,visibility_bucket,visibility_score,page_score
10,1.0,medium,2,2.0
20,1.0,high,4,4.0
25,1.0,high,4,4.0
36,1.0,high,4,4.0
49,1.0,high,4,4.0
55,1.0,high,4,4.0
58,1.0,high,4,4.0
70,1.0,medium,2,2.0
74,2.0,medium,2,4.0
75,1.0,high,4,4.0


In [13]:
step_a["reason_code"] = "ranking_decline"
step_a["action"] = "review for refresh"


step_a

,client_hash_id,content_hash_id,early_clicks,late_clicks,early_impressions,late_impressions,early_avg_position,late_avg_position,early_position_tier,late_position_tier,tier_change,position_change,visibility_bucket,visibility_score,page_score,reason_code,action
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,2.0,0.0,379.0,79.0,4.095154,5.548106,2.0,2.0,0.0,stable,medium,2,0.0,ranking_decline,review for refresh
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,10.0,13.0,2336.0,1607.0,4.319753,4.460415,2.0,2.0,0.0,stable,high,4,0.0,ranking_decline,review for refresh
2,client_65de48885f4ef01b,content_3c286ded8bd68120,7.0,8.0,904.0,1276.0,8.777106,8.135446,2.0,2.0,0.0,stable,high,4,0.0,ranking_decline,review for refresh
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,1.0,7.0,330.0,173.0,5.475501,5.632185,2.0,2.0,0.0,stable,medium,2,0.0,ranking_decline,review for refresh
4,client_65de48885f4ef01b,content_ff867882e604fa96,0.0,0.0,24.0,0.0,2.850000,NaN,1.0,NaN,NaN,NaN,low,1,0.0,ranking_decline,review for refresh
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63851,client_20259bd6705d81d4,content_32058aa8a2e4f4fb,0.0,0.0,0.0,9.0,NaN,2.555556,NaN,1.0,NaN,NaN,medium,2,0.0,ranking_decline,review for refresh
63852,client_20259bd6705d81d4,content_526be944d717ab76,0.0,0.0,0.0,8.0,NaN,26.500000,NaN,4.0,NaN,NaN,low,1,0.0,ranking_decline,review for refresh
63853,client_20259bd6705d81d4,content_20c613ae83bad2ca,0.0,1.0,0.0,64.0,NaN,1.890625,NaN,1.0,NaN,NaN,medium,2,0.0,ranking_decline,review for refresh
63854,client_20259bd6705d81d4,content_d4f53cf222510f09,0.0,0.0,0.0,3.0,NaN,0.666667,NaN,1.0,NaN,NaN,low,1,0.0,ranking_decline,review for refresh


In [14]:
ranked_queue = step_a[step_a["page_score"] > 0][["client_hash_id", "content_hash_id", "page_score", "reason_code","action"]]
ranked_queue = ranked_queue.sort_values("page_score", ascending=False)
ranked_queue = ranked_queue.reset_index(drop=True)
ranked_queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

ranked_queue

OSError: Cannot save file into a non-existent directory: 'work/outputs'

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.